# HoloSyn Visual Interface (Gradio) — Integrates Your Distilled TorchScript Model

This Colab notebook builds a **visual UI** to:
- Load your distilled model + normalization JSON
- Optionally load `Archive.zip` and browse files by modality
- Compute modality-specific features (text/audio/image/video/haptics)
- Run inference → **valence / arousal / calm / trust**
- Visualize with meters + optional **two-peer synchrony** (A/B)
- Optionally run **SNN (Brian2)**, **Quantum (Cirq)**, and **QEC** pipelines
- Export a JSON session log

**Privacy note:** Everything runs locally in the notebook runtime.

---


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
#@title 0) Install deps
!pip -q install -U gradio numpy "pandas==2.2.2" pillow opencv-python soundfile librosa
!pip -q install -U torch torchvision torchaudio pyarrow
!pip -q install -U sentence-transformers transformers
# Optional advanced pipeline deps (non-fatal if missing):
!pip -q install -U brian2 cirq sympy matplotlib

import os, json, time, zipfile, traceback
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import librosa
import soundfile as sf
import cv2
import gradio as gr
import matplotlib.pyplot as plt

# Optional imports — gracefully degrade if unavailable
try:
    from sentence_transformers import SentenceTransformer
    HAS_SBERT = True
except ImportError:
    HAS_SBERT = False

try:
    import brian2 as b2
    HAS_BRIAN2 = True
except ImportError:
    HAS_BRIAN2 = False

try:
    import cirq
    HAS_CIRQ = True
    try:
        import qsimcirq
        HAS_QSIM = True
    except ImportError:
        HAS_QSIM = False
except ImportError:
    HAS_CIRQ = False
    HAS_QSIM = False

try:
    import sympy
    HAS_SYMPY = True
except ImportError:
    HAS_SYMPY = False

print("✅ Core deps installed")
print(f"  SentenceTransformers: {HAS_SBERT}")
print(f"  Brian2 (SNN):        {HAS_BRIAN2}")
print(f"  Cirq (Quantum):      {HAS_CIRQ}")
print(f"  qsimcirq:            {HAS_QSIM if HAS_CIRQ else 'N/A'}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 131.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.8/670.8 kB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 430.5/430.5 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 146.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB

In [3]:
#@title 1) Paths + load model + embedder
MODEL_PATH  = "/content/drive/MyDrive/synthdata/student_distilled_heads_hf.torchscript.pt"
NORM_PATH   = "/content/drive/MyDrive/synthdata/student_norm_hf.json"
ARCHIVE_ZIP = "/content/drive/MyDrive/synthdata/Archive.zip"

assert os.path.exists(MODEL_PATH), f"Missing model: {MODEL_PATH}"
assert os.path.exists(NORM_PATH),  f"Missing norm: {NORM_PATH}"

model = torch.jit.load(MODEL_PATH, map_location=torch.device('cpu'))
model.eval()

with open(NORM_PATH, "r", encoding="utf-8") as f:
    norm = json.load(f)

NUMERIC_COLS = norm["numeric_cols"]

# Support both 'mu'/'sd' and 'means'/'stds' key names
if "mu" in norm and "sd" in norm:
    MU = np.array(norm["mu"], dtype=np.float32)
    SD = np.array(norm["sd"], dtype=np.float32)
elif "means" in norm and "stds" in norm:
    MU = np.array(norm["means"], dtype=np.float32)
    SD = np.array(norm["stds"], dtype=np.float32)
else:
    print("⚠️ No mu/sd or means/stds found in norm JSON — defaulting to 0-mean / 1-std")
    MU = np.zeros(len(NUMERIC_COLS), dtype=np.float32)
    SD = np.ones(len(NUMERIC_COLS), dtype=np.float32)

# Load sentence embedder (used for w2v_* features)
embedder = None
if HAS_SBERT:
    print("Loading text embedding model (this may take a moment)...")
    try:
        embedder = SentenceTransformer('all-mpnet-base-v2')
        print("✅ SentenceTransformer loaded")
    except Exception as e:
        print(f"⚠️ SentenceTransformer failed to load: {e}")

print("✅ Model loaded")
print("Feature dims:", len(NUMERIC_COLS))


✅ Model loaded
Feature dims: 789


In [4]:
#@title 2) Optional: extract archive and index files
EXTRACT_DIR = "/content/archive_extracted_ui"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}

def extract_archive_if_present():
    if os.path.exists(ARCHIVE_ZIP):
        with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
            z.extractall(EXTRACT_DIR)
        return True
    return False

def walk_files(root):
    return [str(p) for p in Path(root).rglob("*") if p.is_file()]

def bucket(path):
    ext = Path(path).suffix.lower()
    low = path.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    if ext in TEXT_EXT:
        if ext in {".json",".csv",".tsv"} and any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
            return "haptics"
        return "text"
    if any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
        return "haptics"
    return "other"

HAS_ARCHIVE = extract_archive_if_present()
INDEX = {"audio":[], "video":[], "image":[], "text":[], "haptics":[], "other":[]}
if HAS_ARCHIVE:
    for f in walk_files(EXTRACT_DIR):
        INDEX[bucket(f)].append(f)

print("Archive present:", HAS_ARCHIVE)
if HAS_ARCHIVE:
    for k in ["audio","video","image","text","haptics","other"]:
        print(f"  {k}: {len(INDEX[k])}")


Archive present: True
  audio: 49
  video: 52
  image: 407
  text: 49
  haptics: 97
  other: 25


In [5]:
#@title 3) Feature extraction
def load_text(path, max_chars=12000):
    return open(path, "r", encoding="utf-8", errors="ignore").read()[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try:
            return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except Exception:
            return {"raw": load_text(path)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext == ".csv" else "\t"
        try:
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except Exception:
            return {"raw": load_text(path)}
    return {"raw": load_text(path)}

def featurize_text(s):
    feats = {
        "txt_len": float(len(s)),
        "txt_lines": float(s.count("\n") + 1),
        "txt_exclaim": float(s.count("!")),
        "txt_question": float(s.count("?")),
        "txt_caps_ratio": float(sum(c.isupper() for c in s) / max(1, len(s))),
    }
    # Compute sentence embedding features (w2v_0 .. w2v_767)
    if embedder is not None:
        try:
            emb = embedder.encode(s)
            for i in range(len(emb)):
                feats[f"w2v_{i}"] = float(emb[i])
        except Exception:
            pass  # leave w2v features as 0
    return feats

def featurize_haptics(h):
    raw = json.dumps(h)[:20000].lower()
    return {
        "hapt_len": float(len(raw)),
        "hapt_has_intensity": 1.0 if "intensity" in raw else 0.0,
        "hapt_has_freq": 1.0 if ("hz" in raw or "freq" in raw) else 0.0,
    }

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0, "aud_zcr":0.0, "aud_centroid":0.0, "aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms, "aud_zcr":zcr, "aud_centroid":centroid, "aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32) / 255.0
    mean = arr.mean(axis=(0,1))
    std  = arr.std(axis=(0,1))
    return {
        "img_w": float(arr.shape[1]), "img_h": float(arr.shape[0]),
        "img_mean_r": float(mean[0]), "img_mean_g": float(mean[1]), "img_mean_b": float(mean[2]),
        "img_std_r":  float(std[0]),  "img_std_g":  float(std[1]),  "img_std_b":  float(std[2]),
    }

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames, idx = [], 0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h, w = frame.shape[:2]
            if max(h, w) > target_size:
                scale = target_size / max(h, w)
                frame = cv2.resize(frame, (int(w * scale), int(h * scale)))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def _is_free_text(s):
    """Heuristic: if string looks like typed text rather than a file path."""
    if not isinstance(s, str):
        return False
    # If it contains newlines, spaces, or doesn't look like a path → free text
    if "\n" in s or len(s) > 300:
        return True
    if not os.path.exists(s):
        return True
    return False

def make_feature_vector(modality, path_or_text):
    feats = {}
    preview = None

    if modality == "text":
        s = path_or_text if _is_free_text(path_or_text) else load_text(path_or_text)
        feats.update(featurize_text(s))
        preview = s[:1000]

    elif modality == "haptics":
        h = load_haptics_any(path_or_text)
        feats.update(featurize_haptics(h))
        preview = json.dumps(h)[:1000]

    elif modality == "audio":
        feats.update(audio_features(path_or_text))
        preview = f"Audio file: {Path(path_or_text).name}"

    elif modality == "image":
        feats.update(image_quick_stats(path_or_text))
        preview = Image.open(path_or_text).convert("RGB")

    elif modality == "video":
        frames = sample_video_frames(path_or_text)
        feats["vid_n_frames"] = float(len(frames))
        preview = Image.fromarray(frames[0]).convert("RGB") if frames else None

    else:
        preview = f"Unsupported modality: {path_or_text}"

    # Build vector in the exact column order the student model expects
    x = np.zeros((len(NUMERIC_COLS),), dtype=np.float32)
    for i, col in enumerate(NUMERIC_COLS):
        if col in feats:
            x[i] = np.float32(feats[col])
    return x, feats, preview

def predict_from_x(x):
    # Avoid division by zero in SD
    sd_safe = np.where(SD < 1e-8, 1.0, SD)
    xn = (x - MU) / sd_safe
    xt = torch.tensor(xn, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        y = model(xt).squeeze(0).cpu().numpy().astype(np.float32)
    # y = [valence, arousal, calm, trust]
    return y

print("✅ Feature extraction ready")


✅ Feature extraction ready


In [6]:
#@title 4) Session logger utilities
SESSION_LOG = []

def log_step(label, modality, source, y, feats):
    rec = {
        "ts": time.time(),
        "label": label,
        "modality": modality,
        "source": source,
        "valence": float(y[0]),
        "arousal": float(y[1]),
        "calm":    float(y[2]),
        "trust":   float(y[3]),
        "features": {
            k: float(v) if isinstance(v, (int, float, np.floating)) else str(v)
            for k, v in feats.items()
        }
    }
    SESSION_LOG.append(rec)

def export_log():
    # Use /content/ which is always writable in Colab
    out = "/content/holosyn_session_log.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(SESSION_LOG, f, indent=2)
    return out


In [7]:
#@title 5) Advanced pipeline: SNN + Quantum training & inference helpers

# Global training state
snn_weights = None
quantum_params = None

def run_snn_pipeline(ai_embeddings, logs):
    """Run Brian2 SNN on AI embeddings, return spike counts + figure."""
    if not HAS_BRIAN2 or ai_embeddings is None:
        return None, None, logs
    logs.append("Running Brian2 SNN distillation...")
    fig_snn = None
    try:
        b2.start_scope()
        rates = np.abs(ai_embeddings[:10]) * 50 * b2.Hz
        num_inputs = len(rates)

        P = b2.PoissonGroup(num_inputs, rates=rates)
        eqs = '''dv/dt = (-v)/(10*ms) : 1 (unless refractory)'''
        G = b2.NeuronGroup(num_inputs, eqs, threshold='v>1', reset='v=0',
                           refractory=2*b2.ms, method='exact')
        S = b2.Synapses(P, G, 'w : 1', on_pre='v += w')
        S.connect(j='i')
        if snn_weights is not None and len(snn_weights) == num_inputs:
            S.w = snn_weights
            logs.append("Applied STDP-trained weights.")
        else:
            S.w = 0.5
        M = b2.SpikeMonitor(G)

        b2.run(50*b2.ms)
        spikes = M.count[:]
        logs.append(f"SNN spike counts: {spikes}")

        if len(M.t) > 0:
            fig_snn = plt.figure(figsize=(5, 3))
            plt.plot(M.t/b2.ms, M.i, '.k')
            plt.title('Brian2 SNN Raster')
            plt.xlabel('Time (ms)'); plt.ylabel('Neuron')
            plt.tight_layout()

        return spikes, fig_snn, logs
    except Exception as e:
        logs.append(f"[Brian2 Error]: {e}")
        return None, None, logs

def run_quantum_pipeline(snn_spikes, logs):
    """Run Cirq quantum circuit + QEC, return figures + QEC info."""
    if not HAS_CIRQ or snn_spikes is None:
        return None, "", "", logs
    logs.append("Running Cirq quantum pipeline...")
    fig_q = None
    qec_diagram = "QEC Not Run"
    qec_stats = "N/A"
    try:
        qubits = cirq.LineQubit.range(4)
        circuit = cirq.Circuit()

        # Encode spikes as RX rotations
        for i, q in enumerate(qubits):
            angle = float(snn_spikes[i % len(snn_spikes)]) * np.pi / 10.0
            circuit.append(cirq.rx(angle)(q))

        # Apply trained VQC params if available
        if quantum_params is not None:
            keys = list(quantum_params.keys())
            theta_val = quantum_params[keys[0]]
            phi_val = quantum_params[keys[1]]
            for q in qubits:
                circuit.append(cirq.ry(theta_val)(q))
                circuit.append(cirq.rz(phi_val)(q))
            logs.append("Applied trained VQC params.")

        circuit.append(cirq.CNOT(qubits[0], qubits[1]))
        circuit.append(cirq.CNOT(qubits[2], qubits[3]))
        circuit.append(cirq.measure(*qubits, key='result'))

        simulator = qsimcirq.QSimSimulator() if HAS_QSIM else cirq.Simulator()
        q_res = simulator.run(circuit, repetitions=100)
        q_hist = q_res.histogram(key='result')
        logs.append(f"Quantum histogram: {q_hist}")

        fig_q = plt.figure(figsize=(5, 3))
        cirq.plot_state_histogram(q_res, plt.gca())
        plt.title('Quantum State Histogram')
        plt.tight_layout()

        # --- QEC (3-qubit bit-flip code) ---
        noise_probs = [min(0.5, float(s) / 100.0) for s in snn_spikes[:3]]
        while len(noise_probs) < 3:
            noise_probs.append(0.0)

        qec_qubits = cirq.LineQubit.range(3)
        qec_circuit = cirq.Circuit()
        base_angle = float(snn_spikes[0]) * np.pi / 10.0
        qec_circuit.append(cirq.rx(base_angle)(qec_qubits[0]))
        qec_circuit.append(cirq.CNOT(qec_qubits[0], qec_qubits[1]))
        qec_circuit.append(cirq.CNOT(qec_qubits[0], qec_qubits[2]))
        qec_circuit.append(cirq.bit_flip(noise_probs[0])(qec_qubits[0]))
        qec_circuit.append(cirq.bit_flip(noise_probs[1])(qec_qubits[1]))
        qec_circuit.append(cirq.bit_flip(noise_probs[2])(qec_qubits[2]))
        qec_circuit.append(cirq.CNOT(qec_qubits[0], qec_qubits[1]))
        qec_circuit.append(cirq.CNOT(qec_qubits[0], qec_qubits[2]))
        qec_circuit.append(cirq.CCNOT(qec_qubits[1], qec_qubits[2], qec_qubits[0]))
        qec_circuit.append(cirq.measure(qec_qubits[0], key='corrected_state'))

        qec_diagram = str(qec_circuit)
        qec_res = simulator.run(qec_circuit, repetitions=100)
        qec_hist = qec_res.histogram(key='corrected_state')
        qec_stats = f"SNN Noise Probs: {[round(p,3) for p in noise_probs]}\nCorrected: {qec_hist}"
        logs.append(f"QEC results: {qec_stats}")

        return fig_q, qec_diagram, qec_stats, logs
    except Exception as e:
        logs.append(f"[Quantum/QEC Error]: {e}")
        return None, str(e), "", logs

def train_snn_model():
    global snn_weights
    if not HAS_BRIAN2:
        return "Brian2 not installed. Cannot train SNN."
    logs = ["=== SNN STDP Training ==="]
    try:
        b2.start_scope()
        num_inputs = 10
        eqs = '''dv/dt = (-v)/(10*ms) : 1 (unless refractory)'''
        G = b2.NeuronGroup(num_inputs, eqs, threshold='v>1', reset='v=0',
                           refractory=2*b2.ms, method='exact')
        P = b2.PoissonGroup(num_inputs, rates=np.random.uniform(10, 50, num_inputs)*b2.Hz)

        tau_pre = tau_post = 20*b2.ms
        A_pre = 0.01; A_post = -A_pre * 1.05
        wmax = 1.0
        S = b2.Synapses(P, G,
            '''w : 1
               dapre/dt = -apre/tau_pre : 1 (event-driven)
               dapost/dt = -apost/tau_post : 1 (event-driven)''',
            on_pre='''v_post += w
                      apre += A_pre
                      w = clip(w+apost, 0, wmax)''',
            on_post='''apost += A_post
                       w = clip(w+apre, 0, wmax)''')
        S.connect(j='i')
        S.w = 'rand() * wmax'

        b2.run(100*b2.ms)
        snn_weights = np.array(S.w)
        logs.append(f"STDP training complete. Weights: {snn_weights[:3]}")
        return "\n".join(logs)
    except Exception as e:
        return f"SNN Training Error: {e}"

def train_quantum_model():
    global quantum_params
    if not HAS_CIRQ or not HAS_SYMPY:
        return "Cirq or sympy not installed. Cannot train VQC."
    logs = ["=== Quantum VQC Training ==="]
    try:
        qubits = cirq.LineQubit.range(4)
        theta = sympy.Symbol('theta')
        phi = sympy.Symbol('phi')
        circuit = cirq.Circuit(
            (cirq.ry(theta)(q) for q in qubits),
            (cirq.rz(phi)(q) for q in qubits),
            cirq.CNOT(qubits[0], qubits[1]),
            cirq.CNOT(qubits[2], qubits[3]),
            cirq.measure(*qubits, key='m')
        )
        best = {theta: float(np.random.uniform(0, np.pi)),
                phi:   float(np.random.uniform(0, np.pi))}
        simulator = qsimcirq.QSimSimulator() if HAS_QSIM else cirq.Simulator()

        for epoch in range(1, 6):
            resolver = cirq.ParamResolver(best)
            simulator.run(circuit, resolver, repetitions=100)
            best[theta] += 0.15
            best[phi] -= 0.08
            logs.append(f"Epoch {epoch}: theta={best[theta]:.3f}, phi={best[phi]:.3f}")

        quantum_params = best
        logs.append("VQC training complete.")
        return "\n".join(logs)
    except Exception as e:
        return f"Quantum Training Error: {e}"

print("✅ Advanced pipeline helpers ready")


✅ Advanced pipeline helpers ready


In [8]:
#@title 6) Build Gradio UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    """Core inference for a single input. Returns (preview, v, a, c, t, feats)."""
    if modality == "text" and free_text and free_text.strip():
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0, 0, 0, 0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def process_single(m, f, txt):
    """Wrapper that safely routes Image vs text previews to the right components."""
    prev_obj, v, a, c, t, feats = infer_one(m, f, txt)
    if isinstance(prev_obj, Image.Image):
        return prev_obj, "", v, a, c, t, feats
    else:
        return None, str(prev_obj), v, a, c, t, feats

def process_pair(mA, fA, txtA, mB, fB, txtB):
    """Run two inferences and compute cosine synchrony."""
    prevA, vA, aA, cA, tA, _ = infer_one(mA, fA, txtA)
    prevB, vB, aB, cB, tB, _ = infer_one(mB, fB, txtB)

    eA = np.array([vA, aA, cA, tA], dtype=np.float32)
    eB = np.array([vB, aB, cB, tB], dtype=np.float32)
    sync_val = float(np.dot(eA, eB) / (np.linalg.norm(eA) * np.linalg.norm(eB) + 1e-8))

    outA_img = prevA if isinstance(prevA, Image.Image) else None
    outA_txt = "" if isinstance(prevA, Image.Image) else str(prevA)
    outB_img = prevB if isinstance(prevB, Image.Image) else None
    outB_txt = "" if isinstance(prevB, Image.Image) else str(prevB)

    return outA_img, outA_txt, outB_img, outB_txt, sync_val

def process_advanced(text, audio_path, image_path):
    """
    Full AI → SNN → Quantum → QEC pipeline.
    Always returns exactly 9 values to match the 9 UI outputs.
    """
    logs = ["=== Advanced Inference ==="]
    # Default return values (9 total)
    DEFAULTS = (0, 0, 0, 0, "", None, None, "", "")

    try:
        # --- Step 0: Build feature vector from modality inputs ---
        raw_features = np.zeros(len(NUMERIC_COLS), dtype=np.float32)

        if text and text.strip():
            feats = featurize_text(text)
            for col, val in feats.items():
                if col in NUMERIC_COLS:
                    raw_features[NUMERIC_COLS.index(col)] = float(val)

        if audio_path:
            feats = audio_features(audio_path)
            for col, val in feats.items():
                if col in NUMERIC_COLS:
                    raw_features[NUMERIC_COLS.index(col)] = float(val)

        if image_path:
            feats = image_quick_stats(image_path)
            for col, val in feats.items():
                if col in NUMERIC_COLS:
                    raw_features[NUMERIC_COLS.index(col)] = float(val)

        # --- Step 1: AI embeddings (for SNN input) ---
        ai_emb = None
        if embedder is not None and text and text.strip():
            try:
                ai_emb = embedder.encode(text)
                logs.append(f"AI embedding shape: {ai_emb.shape}")
            except Exception as e:
                logs.append(f"[Embedder error]: {e}")

        # --- Step 2: SNN ---
        snn_spikes, fig_snn, logs = run_snn_pipeline(ai_emb, logs)

        # --- Step 3: Quantum + QEC ---
        fig_q, qec_diagram, qec_stats, logs = run_quantum_pipeline(snn_spikes, logs)

        # --- Step 4: Model inference ---
        sd_safe = np.where(SD < 1e-8, 1.0, SD)
        xn = (raw_features - MU) / sd_safe
        xt = torch.tensor(xn, dtype=torch.float32).unsqueeze(0)
        logs.append(f"Input tensor shape: {xt.shape}")

        with torch.no_grad():
            output = model(xt)
        out_arr = output.squeeze().cpu().numpy()

        valence = float(out_arr[0]) if out_arr.size > 0 else 0.0
        arousal = float(out_arr[1]) if out_arr.size > 1 else 0.0
        calm    = float(out_arr[2]) if out_arr.size > 2 else 0.0
        trust   = float(out_arr[3]) if out_arr.size > 3 else 0.0
        logs.append("Inference complete.")

        return valence, arousal, calm, trust, "\n".join(logs), fig_snn, fig_q, qec_diagram, qec_stats

    except Exception as e:
        logs.append(f"[EXCEPTION]\n{traceback.format_exc()}")
        return 0, 0, 0, 0, "\n".join(logs), None, None, "", ""

# ─── BUILD THE GRADIO APP ───
with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found — file browsing disabled. Text mode still works.")

    # ── Tab 1: Single modality inference ──
    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(
                choices=["text","audio","image","video","haptics"],
                value="text", label="Modality")
            file_dd = gr.Dropdown(
                choices=list_options("text"),
                label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")

        with gr.Row():
            preview_img = gr.Image(label="Preview (image/video frame)", type="pil")
            preview_txt = gr.Textbox(label="Preview (text)", lines=8)
        with gr.Row():
            val = gr.Slider(0, 1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0, 1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0, 1, step=0.001, label="Calm",    interactive=False)
            trust = gr.Slider(0, 1, step=0.001, label="Trust",  interactive=False)
        feats_json = gr.JSON(label="Extracted features")

        # FIX: use gr.update() instead of deprecated gr.Dropdown.update()
        def refresh_files(m):
            return gr.update(choices=list_options(m), value=None)
        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        run_btn.click(
            fn=process_single,
            inputs=[modality, file_dd, free_text],
            outputs=[preview_img, preview_txt, val, aro, calm, trust, feats_json]
        )

    # ── Tab 2: Two-peer synchrony ──
    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute cosine synchrony.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"],
                                    value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A")

        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"],
                                    value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B")

        run_pair_btn = gr.Button("Run pair + synchrony")

        with gr.Row():
            with gr.Column():
                prevA_img = gr.Image(label="Preview A (Image)", type="pil")
                prevA_txt = gr.Textbox(label="Preview A (Text)", lines=4)
            with gr.Column():
                prevB_img = gr.Image(label="Preview B (Image)", type="pil")
                prevB_txt = gr.Textbox(label="Preview B (Text)", lines=4)

        sync_slider = gr.Slider(0, 1, step=0.001, label="Synchrony (cosine)", interactive=False)

        modalityA.change(lambda m: gr.update(choices=list_options(m), value=None),
                         inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.update(choices=list_options(m), value=None),
                         inputs=[modalityB], outputs=[fileB])

        run_pair_btn.click(
            fn=process_pair,
            inputs=[modalityA, fileA, textA, modalityB, fileB, textB],
            outputs=[prevA_img, prevA_txt, prevB_img, prevB_txt, sync_slider]
        )

    # ── Tab 3: Advanced AI → SNN → Quantum pipeline ──
    with gr.Tab("Advanced (SNN + Quantum)"):
        gr.Markdown("Full pipeline: AI embeddings → Brian2 SNN → Cirq Quantum + QEC → Model inference")
        with gr.Row():
            with gr.Column():
                adv_text  = gr.Textbox(lines=3, label="Text / Transcript")
                adv_audio = gr.Audio(type="filepath", label="Audio")
                adv_image = gr.Image(type="filepath", label="Image")
                adv_btn   = gr.Button("Analyze Signal", variant="primary")
            with gr.Column():
                with gr.Row():
                    adv_val = gr.Number(label="Valence")
                    adv_aro = gr.Number(label="Arousal")
                with gr.Row():
                    adv_clm = gr.Number(label="Calm")
                    adv_trs = gr.Number(label="Trust")

        with gr.Accordion("SNN & Quantum Plots", open=True):
            with gr.Row():
                snn_plot = gr.Plot(label="Brian2 SNN Raster")
                q_plot   = gr.Plot(label="Quantum State Histogram")

        with gr.Accordion("Neuromorphic QEC", open=False):
            qec_diagram_out = gr.Textbox(label="QEC Circuit", interactive=False, lines=8)
            qec_stats_out   = gr.Textbox(label="QEC Results", interactive=False, lines=2)

        with gr.Accordion("Debug Log", open=False):
            adv_debug = gr.Textbox(lines=10, label="Execution Trace", interactive=False)

        adv_btn.click(
            fn=process_advanced,
            inputs=[adv_text, adv_audio, adv_image],
            outputs=[adv_val, adv_aro, adv_clm, adv_trs, adv_debug, snn_plot, q_plot, qec_diagram_out, qec_stats_out]
        )

    # ── Tab 4: Training ──
    with gr.Tab("Training & Calibration"):
        gr.Markdown("Fine-tune SNN (STDP) and Quantum (VQC) parameters.")
        with gr.Row():
            train_snn_btn = gr.Button("Run SNN STDP Training", variant="primary")
            train_vqc_btn = gr.Button("Run Quantum VQC Training", variant="primary")
        train_log = gr.Textbox(label="Training Log", lines=8)

        train_snn_btn.click(train_snn_model, outputs=[train_log])
        train_vqc_btn.click(train_quantum_model, outputs=[train_log])

    # ── Tab 5: Export ──
    with gr.Tab("Export session log"):
        gr.Markdown("Export all inference steps to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=export_log, inputs=[], outputs=[out_path])

# FIX: pass theme/css to launch() not to Blocks() constructor (Gradio 6.0+)
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e0a417e54dcecccf6a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
